#  ⭐ `dim_concentracao`


In [1]:
%run ../../config/bootstrap.py


from pathlib import Path
from utils import get_project_root , normalize_concentracao

import re
import pandas as pd
from unidecode import unidecode

In [2]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
path = Path(project_root) / "data/01_bronze/anvisa/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(Path(project_root) / "data/01_bronze/anvisa/Medicamentos/Medicamentos.parquet")  
bronze['CONCENTRACAO']= bronze['CONCENTRACAO'].fillna('DESCONHECIDO')
bronze = (
    bronze[['CONCENTRACAO']]
    .value_counts()
    .rename('FREQUENCIA_REGISTROS')
    .reset_index()
) 
bronze.columns

Index(['CONCENTRACAO', 'FREQUENCIA_REGISTROS'], dtype='object')

In [4]:
bronze.head()

,CONCENTRACAO,FREQUENCIA_REGISTROS
0,DESCONHECIDO,491187
1,100mg,2557
2,500mg,2279
3,1g,1989
4,500 mg,1510


In [5]:
bronze.CONCENTRACAO.nunique()

14122

In [6]:
bronze.to_csv(Path(project_root) / "eda/Medicamentos/Medicamentos_CONCENTRACAO.csv", sep=";", index=False)

# 🥈 Silver

normalized data, medium quality


In [7]:
silver = bronze.copy()


In [8]:
silver.head(40)

,CONCENTRACAO,FREQUENCIA_REGISTROS
0,DESCONHECIDO,491187
1,100mg,2557
2,500mg,2279
3,1g,1989
4,500 mg,1510
5,100 mg,1360
6,50mg,1295
7,10mg,1256
8,5 mg,1024
9,40mg,1003


In [9]:
silver = normalize_concentracao(silver, col="CONCENTRACAO")

In [10]:
silver.columns

Index(['CONCENTRACAO', 'FREQUENCIA_REGISTROS', 'CONCENTRACAO_TIPO_VALOR',
       'CONCENTRACAO_TIPO_CHAVE', 'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 'CONCENTRACAO_VALOR_DENOMINADOR'],
      dtype='object')

In [11]:
(
    silver.groupby("CONCENTRACAO_TIPO_CHAVE")["FREQUENCIA_REGISTROS"]
    .sum()
    .reset_index(name="FREQUENCIA_REGISTROS_TOTAL")
).sort_values(by='FREQUENCIA_REGISTROS_TOTAL', ascending=False)

,CONCENTRACAO_TIPO_CHAVE,FREQUENCIA_REGISTROS_TOTAL
0,0,499050
1,1,46849
6,6,31856
2,2,7076
5,5,2610
4,4,2192
3,3,740
9,9,657
7,7,362
8,8,65


In [12]:
hist =  silver[['CONCENTRACAO', 'CONCENTRACAO_TIPO_CHAVE',
       'CONCENTRACAO_TIPO_VALOR', 'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 'CONCENTRACAO_VALOR_DENOMINADOR']]


In [13]:
hist.to_parquet(Path(project_root) / "data/02_silver/hist_dim_concentracao/hist_dim_concentracao.parquet", index=False)

# 🥇 Gold


In [14]:
dim = hist[['CONCENTRACAO_TIPO_CHAVE', 'CONCENTRACAO_TIPO_VALOR']].drop_duplicates().sort_values('CONCENTRACAO_TIPO_VALOR')
dim

,CONCENTRACAO_TIPO_CHAVE,CONCENTRACAO_TIPO_VALOR
0,0,desconhecido
3,2,gram (g)
159,9,gram per millilitre (g/mL)
200,3,microgram (ug)
1,1,milligram (mg)
1566,8,milligram per gram (mg/g)
713,7,milligram per milligram (mg/mg)
14,6,milligram per millilitre (mg/mL)
143,4,millilitre (mL)
63,5,percent (%)


In [15]:
dim.to_csv(Path(project_root) / "data/03_gold/dim_concentracao/dim_concentracao.csv", sep=";", index=False)

In [17]:
dim.to_parquet(Path(project_root) / "data/03_gold/dim_concentracao/dim_concentracao.parquet",index=False)